# Multi-building shape and transformation test

This notebook stress-tests the local Team 04 boundary tools for one site with many buildings.

It covers:
1. generating multiple building shapes: `L`, `I`, `Y`, `T`, `H`, `X`, `O`
2. moving each building to a target location
3. orienting or rotating each building
4. mirroring selected buildings
5. checking whether the transformed boundary still fits inside the site

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

workspace_root = Path.cwd().resolve()
candidate_roots = (
    workspace_root,
    workspace_root.parent,
    workspace_root / 'team_04',
    workspace_root.parent / 'team_04',
)
MODULE_ROOT = next((path for path in candidate_roots if (path / 'agent').exists()), None)
if MODULE_ROOT is None:
    raise FileNotFoundError('Run this notebook from the workspace root, the team_04 folder, or the team_04/notebooks folder.')

module_root_str = str(MODULE_ROOT)
if module_root_str not in sys.path:
    sys.path.insert(0, module_root_str)

OUTPUT_JSON = MODULE_ROOT / 'notebooks' / 'multi_building_shape_transform_report.json'

In [ ]:
# Tool imports are loaded in the execution cell below.

In [ ]:
site_boundary = [
    [0.0, 0.0, 0.0],
    [180.0, 0.0, 0.0],
    [180.0, 130.0, 0.0],
    [0.0, 130.0, 0.0],
    [0.0, 0.0, 0.0],
]

building_scenarios = [
    {'label': 'building_01', 'shape': 'I', 'area': 1100.0, 'target': [25.0, 25.0], 'rotation': 0.0, 'mirror': False, 'mirror_axis': 'y'},
    {'label': 'building_02', 'shape': 'L', 'area': 1250.0, 'target': [60.0, 35.0], 'rotation': 18.0, 'mirror': True, 'mirror_axis': 'y'},
    {'label': 'building_03', 'shape': 'Y', 'area': 950.0, 'target': [95.0, 38.0], 'orientation': 32.0, 'mirror': False, 'mirror_axis': 'y'},
    {'label': 'building_04', 'shape': 'T', 'area': 1000.0, 'target': [132.0, 34.0], 'rotation': -12.0, 'mirror': False, 'mirror_axis': 'y'},
    {'label': 'building_05', 'shape': 'H', 'area': 1150.0, 'target': [42.0, 92.0], 'rotation': 10.0, 'mirror': True, 'mirror_axis': 'x'},
    {'label': 'building_06', 'shape': 'X', 'area': 900.0, 'target': [92.0, 92.0], 'orientation': 45.0, 'mirror': False, 'mirror_axis': 'y'},
    {'label': 'building_07', 'shape': 'O', 'area': 850.0, 'target': [172.0, 116.0], 'rotation': 0.0, 'mirror': False, 'mirror_axis': 'y'},
]

site_boundary

In [ ]:
from agent.tools.generate_building_boundary import generate_building_boundary
from agent.tools.modify_building_boundary import modify_building_boundary

results = []
gh_payloads = []

for scenario in building_scenarios:
    generated = generate_building_boundary(
        area=scenario['area'],
        building_type=scenario['shape'],
    )
    modified = modify_building_boundary(
        geometry_id=generated['data']['geometry_id'],
        boundary=generated['data']['boundary'],
        target_centroid_xy=scenario['target'],
        rotation_degrees=scenario.get('rotation', 0.0),
        orientation_degrees=scenario.get('orientation', 0.0),
        apply_mirror=scenario['mirror'],
        mirror_axis=scenario['mirror_axis'],
        site_boundary=site_boundary,
    )

    results.append({
        'label': scenario['label'],
        'shape': scenario['shape'],
        'area': modified['data']['boundary_area_sqm'],
        'centroid': modified['data']['centroid'],
        'fits_within_site_boundary': modified['data']['fits_within_site_boundary'],
        'boundary_intersects_site_boundary': modified['data']['boundary_intersects_site_boundary'],
        'violations': modified['data']['violations'],
    })

    gh_payloads.append({
        'geometry_id': modified['data']['geometry_id'],
        'building_footprint': {
            'type': 'Polygon',
            'coordinates': modified['data']['transformed_boundary'],
        },
        'metadata': {
            'shape_type': scenario['shape'],
            'transform_parameters': modified['data']['transform_parameters'],
            'fits_within_site_boundary': modified['data']['fits_within_site_boundary'],
            'violations': modified['data']['violations'],
        },
    })

results

In [ ]:
inside_site = [item for item in results if item['fits_within_site_boundary']]
outside_site = [item for item in results if not item['fits_within_site_boundary']]

summary = {
    'building_count': len(results),
    'inside_site_count': len(inside_site),
    'outside_site_count': len(outside_site),
    'outside_labels': [item['label'] for item in outside_site],
}
summary

In [ ]:
assert len(results) == len(building_scenarios)
assert all(item['shape'] in {'L', 'I', 'Y', 'T', 'H', 'X', 'O'} for item in results)
assert any(not item['fits_within_site_boundary'] for item in results)
assert any(item['fits_within_site_boundary'] for item in results)

'Notebook checks passed.'

In [ ]:
import json

report = {
    'site_boundary': site_boundary,
    'summary': summary,
    'results': results,
    'grasshopper_payloads': gh_payloads,
}

OUTPUT_JSON.write_text(json.dumps(report, indent=2), encoding='utf-8')
OUTPUT_JSON.name

## Grasshopper follow-up

The Python side now gives you a stable payload for a future Grasshopper `modify_building_boundary_04` tool.

The intended live loop is:
1. generate a footprint in Python
2. modify it in Grasshopper with move, orientation, rotation, or mirroring
3. check whether the transformed boundary leaves the site or crosses the site boundary
4. return the transformed coordinates plus geometry facts to the planner or notebook